# NB1 — BPE Tokenization
### Session 04 · M01 Foundation — Modern LLM Internals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/04_BPE_Temperature_Top_K_Top_P/notebooks/NB1_BPE_Tokenization.ipynb)

---

**The central question this notebook answers:**

> GPT never sees the word "tokenization". It sees a list of integers like `[30001, 1634, 318]`.  
> *How does it decide which integers represent which text?*

The answer is **Byte Pair Encoding (BPE)** — the tokenization algorithm used by GPT-2, GPT-3, GPT-4, LLaMA, Mistral, and almost every modern LLM.

---

**What you will build in this notebook:**
- A BPE tokenizer from scratch using only Python's standard library
- An understanding of why character-level and word-level tokenization fail
- A comparison of GPT-2 vs GPT-4 tokenization using the real `tiktoken` library

**No API key required.** This notebook runs entirely offline.

---

**Four sections:**
| Section | What you do |
|---|---|
| 1 — Concept | Read and run fully working code. Understand BPE step by step. |
| 2 — Guided Build | Fill in the `___` placeholders to implement your own BPE tokenizer. |
| 3 — Experiment | Use `tiktoken` to explore how real LLMs tokenize text, code, and numbers. |
| 4 — Challenge | Tokenize a code snippet with your BPE. Compare with tiktoken. |

In [ ]:
# Install the only external library this notebook needs
# tiktoken is OpenAI's official tokenizer library — we use it in Section 3
!pip install tiktoken --quiet

---
## SECTION 1 — CONCEPT
### Why tokenization exists, why simple approaches fail, and how BPE works

> Run every cell in this section. Nothing to fill in — just read, run, and understand.

### 1.1 — Why does tokenization exist at all?

Neural networks can only process numbers — not text. So every piece of text must be converted to a sequence of integers before the model sees it.

**Analogy:** Imagine you need to send a letter by telegraph but the machine only transmits numbers. You and the receiver agree on a codebook ahead of time: `A=1, B=2, C=3 ...` etc. Tokenization is that codebook.

The codebook (vocabulary) must be:
- **Fixed size** — the model's embedding table has a fixed number of rows
- **Decided before training** — the same codebook is used for training and inference
- **Covering** — able to represent any text the model might encounter

The question is: *what are the units in the codebook — characters, words, or something in between?*

In [ ]:
# === APPROACH 1: Character-level tokenization ===
# The simplest idea: every character is a token

text = "low lower newest widest"

# Build a character-level vocabulary
char_vocab = sorted(set(text))          # all unique characters, sorted
char_to_id = {ch: i for i, ch in enumerate(char_vocab)}  # map char → integer ID

# Tokenize by splitting into individual characters
char_tokens = list(text)  # ['l', 'o', 'w', ' ', 'l', 'o', 'w', 'e', 'r', ...]
char_ids    = [char_to_id[ch] for ch in char_tokens]  # each char → its ID

print("=== Character-level tokenization ===")
print(f"Text: '{text}'")
print(f"Vocabulary size: {len(char_vocab)} characters")
print(f"Vocabulary: {char_vocab}")
print(f"Token sequence: {char_tokens}")
print(f"ID sequence:    {char_ids}")
print(f"Sequence length: {len(char_ids)} tokens for {len(text.split())} words")
print()
print("PROBLEM: 'low' and 'lower' share the characters l-o-w,")
print("         but a character-level model must learn that relationship")
print("         from scratch over many layers — very hard for short sequences.")

In [ ]:
# === APPROACH 2: Word-level tokenization ===
# Treat every word as a token — the opposite extreme

# Imagine a corpus of English text
corpus_words = [
    "low", "lower", "lowest",
    "new", "newer", "newest",
    "wide", "wider", "widest",
    "tokenization", "tokenizing", "tokenized",  # variations of one word
    "GPT", "GPT-2", "GPT-3", "GPT-4",           # model names
    "#python", "@user", "https://example.com"    # social media / URLs
]

word_vocab = {word: i for i, word in enumerate(sorted(set(corpus_words)))}

# What happens to words NOT in the vocabulary?
unseen_words = ["tokenizer", "GPT-5", "OpenAI"]

print("=== Word-level tokenization ===")
print(f"Vocabulary has {len(word_vocab)} entries (just from our tiny example)")
print(f"Real English has ~170,000 words + proper nouns + URLs + code + ...")
print(f"GPT-3's actual vocabulary: 50,257 tokens")
print()

print("PROBLEM 1 — Vocabulary explosion:")
print("  'low', 'lower', 'lowest', 'newer', 'newest', 'wider', 'widest'")
print("  Each variation needs its own entry — but they share morphology!")
print()

print("PROBLEM 2 — Out-of-vocabulary (OOV) words:")
for word in unseen_words:
    if word in word_vocab:
        print(f"  '{word}' → ID {word_vocab[word]}  ✓ found")
    else:
        print(f"  '{word}' → <UNK>  ✗ NOT in vocab — model sees nothing useful")

print()
print("With word-level tokenization, 'tokenizer' is just a mystery box to the model.")
print("It can't use the fact that 'tokenizer' contains 'token' + 'izer'.")

### 1.2 — The BPE solution: start with characters, merge up

Byte Pair Encoding was invented in the 1990s for data compression. It was adapted for NLP in 2016, then GPT-2 made it the standard.

**The idea:**
1. Start with a vocabulary of individual characters (solves the OOV problem — any text can be represented)
2. Scan the training corpus and count every adjacent pair of symbols
3. Merge the most frequent pair into a new single symbol
4. Repeat steps 2–3 until the vocabulary reaches the target size

After training, common words become single tokens. Rare words are split into familiar subword pieces.

> **Example:** `"tokenization"` → `["token", "ization"]`  
> `"tokenizing"` → `["token", "iz", "ing"]`  
> Both share `"token"` — the model can use what it knows about `"token"` for both words.

In [ ]:
# ============================================================
# BPE FROM SCRATCH — complete, working implementation
# We'll use the classic example from the original BPE paper
# ============================================================

from collections import defaultdict

# --- Step 0: Represent the corpus as character sequences ---
# Each word is split into characters, with a special end-of-word marker '</w>'
# '</w>' lets the model know where word boundaries are
# The numbers represent word frequency in the corpus

def get_initial_vocab(corpus_freq):
    """Convert a word-frequency dict into BPE's internal representation.
    
    Each word becomes a tuple of its characters (plus an end-of-word marker).
    Example: 'low' (freq=5) → ('l', 'o', 'w', '</w>'): 5
    """
    vocab = {}
    for word, freq in corpus_freq.items():
        # Split the word into individual characters and add the end marker
        char_sequence = tuple(list(word) + ['</w>'])  # e.g. ('l','o','w','</w>')
        vocab[char_sequence] = freq
    return vocab


def get_pair_counts(vocab):
    """Count how often each adjacent pair of symbols appears across the corpus.
    
    If the vocab entry ('l','o','w','</w>') appears 5 times,
    it contributes 5 counts to ('l','o'), 5 to ('o','w'), 5 to ('w','</w>').
    """
    pairs = defaultdict(int)  # default count is 0
    
    for char_sequence, freq in vocab.items():
        # Look at each adjacent pair in this word's character sequence
        for i in range(len(char_sequence) - 1):
            pair = (char_sequence[i], char_sequence[i + 1])  # e.g. ('l', 'o')
            pairs[pair] += freq  # weight by word frequency
    
    return pairs


def merge_pair(vocab, best_pair):
    """Merge all occurrences of best_pair into a single new symbol.
    
    Example: if best_pair = ('e', 's') and we have ('n','e','w','e','s','t','</w>'),
    the result is ('n','e','w','es','t','</w>')
    """
    new_vocab = {}
    
    for char_sequence, freq in vocab.items():
        new_sequence = []  # we'll rebuild the sequence
        i = 0
        while i < len(char_sequence):
            # Check if positions i and i+1 match the pair we want to merge
            if i < len(char_sequence) - 1 and char_sequence[i] == best_pair[0] and char_sequence[i+1] == best_pair[1]:
                merged = best_pair[0] + best_pair[1]  # e.g. 'e' + 's' = 'es'
                new_sequence.append(merged)
                i += 2  # skip both characters — they've been merged
            else:
                new_sequence.append(char_sequence[i])
                i += 1
        new_vocab[tuple(new_sequence)] = freq
    
    return new_vocab


# --- The corpus: word → frequency in training text ---
# This is the exact example from the original BPE paper (Sennrich et al., 2016)
corpus = {
    'low':    5,
    'lower':  2,
    'newest': 6,
    'widest': 3,
}

# Convert to BPE internal format
vocab = get_initial_vocab(corpus)

print("=== Initial vocabulary (character sequences) ===")
print("Format: (character sequence) → frequency")
print()
for seq, freq in vocab.items():
    print(f"  {seq} → {freq}")

print()
print(f"Initial symbol inventory: {sorted(set(sym for seq in vocab for sym in seq))}")

In [ ]:
# --- Run BPE for N_MERGES rounds ---
# Each round merges one pair and grows the vocabulary by one symbol

N_MERGES = 8  # how many merge operations to perform
merge_rules = []  # keep track of what was merged in what order

print("=== BPE merge rounds ===")
print()

for merge_step in range(1, N_MERGES + 1):
    # Count all adjacent pairs in the current vocabulary
    pairs = get_pair_counts(vocab)
    
    if not pairs:
        print("No more pairs to merge.")
        break
    
    # Find the most frequent pair — this is the pair we will merge
    best_pair = max(pairs, key=lambda p: pairs[p])
    best_count = pairs[best_pair]
    
    # Record this merge rule
    merge_rules.append(best_pair)
    
    print(f"Round {merge_step}: merge {best_pair} (appears {best_count} times)")
    print(f"  New token created: '{best_pair[0] + best_pair[1]}'")
    
    # Apply the merge to the entire vocabulary
    vocab = merge_pair(vocab, best_pair)
    
    # Show the updated vocabulary
    print(f"  Vocabulary after merge:")
    for seq, freq in vocab.items():
        print(f"    {seq} → {freq}")
    print()

print("=== Final merge rules ===")
for i, rule in enumerate(merge_rules, 1):
    print(f"  Rule {i}: merge '{rule[0]}' + '{rule[1]}' → '{rule[0]+rule[1]}'")

In [ ]:
# --- Apply learned merge rules to new text ---
# This is what happens at inference time: we apply the rules in the order they were learned

def apply_bpe_rules(word, merge_rules):
    """Tokenize a single word by applying BPE merge rules in order.
    
    We always apply rules in the same order they were learned during training.
    This ensures the tokenization is deterministic.
    """
    # Start with the word split into individual characters + end marker
    symbols = list(word) + ['</w>']
    
    for rule in merge_rules:
        # Try to apply this merge rule anywhere in the current sequence
        new_symbols = []
        i = 0
        while i < len(symbols):
            # Check if this position matches the rule
            if i < len(symbols) - 1 and symbols[i] == rule[0] and symbols[i+1] == rule[1]:
                new_symbols.append(rule[0] + rule[1])  # merge them
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    
    return symbols


# Test on words from the corpus and new words not in the corpus
test_words = ['low', 'lower', 'newest', 'widest', 'lowest', 'newer', 'wide']

print("=== Applying BPE rules to words ===")
print()
for word in test_words:
    tokens = apply_bpe_rules(word, merge_rules)
    in_corpus = '(in corpus)' if word in corpus else '(NEW - never seen during training)'
    print(f"  '{word}' {in_corpus}")
    print(f"    → {tokens}")
    print()

print("Key observation:")
print("  'low', 'lower', 'lowest' all start with the same token 'low</w>' or 'low'")
print("  The model can use what it knows about 'low' for all three words.")
print("  Even 'lowest' — which was NOT in the training corpus — is handled gracefully.")

---
## SECTION 2 — GUIDED BUILD
### Implement your own BPE vocabulary builder and tokenizer

> **Instructions:** Fill in every `___` placeholder. Do not change any other part of the code.  
> The code structure and function signatures are already correct — you are filling in the logic.

In [ ]:
# === TODO 1: Implement build_bpe_vocab() ===
# 
# This function runs the full BPE training process:
# 1. Convert the corpus to character sequences
# 2. Repeatedly find the most frequent pair and merge it
# 3. Return the list of merge rules and the final symbol inventory
#
# HINT: Look at how get_initial_vocab(), get_pair_counts(), and merge_pair()
#       work in Section 1. Call them in the right order inside the loop.

from collections import defaultdict

# Copy the helper functions from Section 1 so this cell is self-contained
def get_initial_vocab(corpus_freq):
    vocab = {}
    for word, freq in corpus_freq.items():
        char_sequence = tuple(list(word) + ['</w>'])
        vocab[char_sequence] = freq
    return vocab

def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for char_sequence, freq in vocab.items():
        for i in range(len(char_sequence) - 1):
            pair = (char_sequence[i], char_sequence[i + 1])
            pairs[pair] += freq
    return pairs

def merge_pair(vocab, best_pair):
    new_vocab = {}
    for char_sequence, freq in vocab.items():
        new_sequence = []
        i = 0
        while i < len(char_sequence):
            if i < len(char_sequence) - 1 and char_sequence[i] == best_pair[0] and char_sequence[i+1] == best_pair[1]:
                new_sequence.append(best_pair[0] + best_pair[1])
                i += 2
            else:
                new_sequence.append(char_sequence[i])
                i += 1
        new_vocab[tuple(new_sequence)] = freq
    return new_vocab


def build_bpe_vocab(corpus_freq, num_merges):
    """Train a BPE tokenizer on a word-frequency corpus.
    
    Args:
        corpus_freq: dict mapping word → frequency, e.g. {'low': 5, 'lower': 2}
        num_merges:  how many merge operations to perform
    
    Returns:
        merge_rules: list of (sym1, sym2) pairs in the order they were merged
        final_vocab: the vocab dict after all merges
    """
    # Step 1: Convert the corpus to initial character-level sequences
    vocab = ___(corpus_freq)  # TODO: call the right function
    
    merge_rules = []  # we'll collect the merge rules here
    
    # Step 2: Repeat for num_merges rounds
    for _ in range(num_merges):
        # Count all adjacent pairs
        pairs = ___(vocab)  # TODO: call the right function
        
        # If there are no pairs left to merge, stop early
        if not pairs:
            break
        
        # Find the most frequent pair
        # HINT: use max() with key=lambda p: pairs[p]
        best_pair = max(___, key=lambda p: ___[p])  # TODO: fill in the two gaps
        
        # Record this rule
        merge_rules.append(best_pair)
        
        # Apply the merge to update the vocabulary
        vocab = ___(vocab, best_pair)  # TODO: call the right function
    
    return merge_rules, vocab


# --- Test your implementation ---
test_corpus = {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}

rules, final_vocab = build_bpe_vocab(test_corpus, num_merges=8)

print("=== Your BPE training results ===")
print(f"Number of merge rules learned: {len(rules)}")
print()
print("Merge rules (in order):")
for i, rule in enumerate(rules, 1):
    print(f"  {i}. '{rule[0]}' + '{rule[1]}' → '{rule[0]+rule[1]}'")
print()
print("Final vocabulary:")
for seq, freq in final_vocab.items():
    print(f"  {seq} → {freq}")

In [ ]:
# === TODO 2: Implement tokenize_with_bpe() ===
#
# This function tokenizes a full sentence using the merge rules you just learned.
# It must:
#   1. Split the sentence into words
#   2. Tokenize each word by applying all merge rules in order
#   3. Return the flat list of tokens for the whole sentence
#
# HINT: look at apply_bpe_rules() in Section 1 — you can call it here directly.

def apply_bpe_rules(word, merge_rules):
    """Helper from Section 1 — tokenizes a single word."""
    symbols = list(word) + ['</w>']
    for rule in merge_rules:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == rule[0] and symbols[i+1] == rule[1]:
                new_symbols.append(rule[0] + rule[1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def tokenize_with_bpe(text, merge_rules):
    """Tokenize a full sentence using learned BPE merge rules.
    
    Args:
        text:        a string, e.g. 'the newest and widest low'
        merge_rules: list of (sym1, sym2) tuples, in training order
    
    Returns:
        A flat list of BPE tokens for the whole sentence.
    """
    all_tokens = []   # we'll collect tokens from every word here
    
    # Step 1: split the input text into words
    # HINT: str has a built-in method for this
    words = text.___()  # TODO: split on whitespace
    
    # Step 2: tokenize each word and collect the results
    for word in ___:  # TODO: iterate over the words list
        # Tokenize this word using the merge rules
        word_tokens = apply_bpe_rules(___, ___)  # TODO: pass word and merge_rules
        
        # Add this word's tokens to the running list
        all_tokens.___(word_tokens)  # TODO: use extend (not append) to flatten the list
    
    return all_tokens


# --- Test your tokenizer ---
test_sentences = [
    "the newest widest low",
    "lower and newest",
    "widest lower low newest",
]

print("=== Your BPE tokenizer results ===")
print()
for sentence in test_sentences:
    tokens = tokenize_with_bpe(sentence, rules)
    print(f"Input:  '{sentence}'")
    print(f"Tokens: {tokens}")
    print(f"Count:  {len(tokens)} tokens for {len(sentence.split())} words")
    print()

---
## SECTION 3 — EXPERIMENT
### Real-world tokenization with tiktoken

Now we switch from our toy implementation to the real thing.  
`tiktoken` is the official library OpenAI built to run the same BPE algorithm that GPT-2, GPT-3, and GPT-4 use.

We'll explore:
- How different encodings (GPT-2 vs GPT-4) tokenize the same text differently
- How tokenization affects text in different languages
- The "tokenization affects cost" effect — why some prompts are more expensive

In [ ]:
import tiktoken

# tiktoken supports multiple encodings — each corresponds to a model family:
#   'gpt2'         → GPT-2 (50,257 tokens)
#   'p50k_base'    → GPT-3 (50,281 tokens)
#   'cl100k_base'  → GPT-3.5 and GPT-4 (100,277 tokens)
#   'o200k_base'   → GPT-4o (200,019 tokens)

enc_gpt2 = tiktoken.get_encoding('gpt2')           # GPT-2 encoding
enc_gpt4 = tiktoken.get_encoding('cl100k_base')    # GPT-4 encoding

print("=== tiktoken is ready ===")
print(f"GPT-2 vocabulary size:  {enc_gpt2.n_vocab:,}")
print(f"GPT-4 vocabulary size:  {enc_gpt4.n_vocab:,}")
print()
print("Why does GPT-4 have ~2x the vocabulary? More vocab = fewer tokens per word.")
print("That means shorter sequences = lower memory cost = can fit more context.")

In [ ]:
# === Experiment 3.1 — Same text, different encodings ===

def compare_encodings(text):
    """Show how GPT-2 and GPT-4 encodings tokenize the same text."""
    tokens_gpt2 = enc_gpt2.encode(text)   # list of integer IDs
    tokens_gpt4 = enc_gpt4.encode(text)   # list of integer IDs
    
    # Decode each token back to the text it represents
    pieces_gpt2 = [enc_gpt2.decode([t]) for t in tokens_gpt2]  # each ID → its string
    pieces_gpt4 = [enc_gpt4.decode([t]) for t in tokens_gpt4]
    
    print(f"Text: '{text}'")
    print(f"  GPT-2 ({len(tokens_gpt2):3d} tokens): {pieces_gpt2}")
    print(f"  GPT-4 ({len(tokens_gpt4):3d} tokens): {pieces_gpt4}")
    print()

# Test with various inputs
test_texts = [
    "ChatGPT is an artificial intelligence chatbot",
    "tokenization",               # single word — how does it split?
    "tokenizer tokenizing tokenized",  # word family — do they share prefixes?
    "https://www.example.com/path?query=value",  # URL — usually expensive
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

print("=== Encoding comparison: GPT-2 vs GPT-4 ===")
print()
for text in test_texts:
    compare_encodings(text)

In [ ]:
# === Experiment 3.2 — How different languages tokenize ===
# This is where tokenization really affects cost!
# GPT-4's vocabulary was trained mostly on English text.
# Non-English text often requires more tokens per word.

multilingual_texts = [
    ("English",   "The quick brown fox jumps over the lazy dog"),
    ("Spanish",   "El zorro marrón rápido salta sobre el perro perezoso"),
    ("French",    "Le rapide renard brun saute par-dessus le chien paresseux"),
    ("German",    "Der schnelle braune Fuchs springt über den faulen Hund"),
    ("Arabic",    "الثعلب البني السريع يقفز فوق الكلب الكسول"),
    ("Chinese",   "敏捷的棕色狐狸跳过了懒狗"),
    ("Hindi",     "तेज़ भूरी लोमड़ी आलसी कुत्ते के ऊपर कूदती है"),
]

print("=== Token efficiency by language (GPT-4 cl100k_base) ===")
print()
print(f"{'Language':<10} {'Tokens':>6} {'Words':>6} {'Tokens/Word':>12}")
print("-" * 40)

for language, text in multilingual_texts:
    tokens = enc_gpt4.encode(text)
    words = len(text.split())          # rough word count (doesn't work for Chinese, but illustrative)
    ratio = len(tokens) / max(words, 1)
    print(f"{language:<10} {len(tokens):>6} {words:>6} {ratio:>12.2f}")

print()
print("Why does this matter for you?")
print("API pricing is per token, not per word.")
print("A prompt in Arabic may cost 2-4x more than the same content in English.")
print("When you build multilingual apps, always estimate token cost per language.")

In [ ]:
# === Experiment 3.3 — Numbers and code tokenization ===
# Numbers are a known pain point in LLM tokenization:
# - Each digit may be its own token
# - This is why LLMs struggle with arithmetic!

number_examples = [
    "42",
    "1000",
    "1000000",
    "3.14159",
    "2024-05-11",     # date in ISO format
    "$1,234,567.89",  # formatted number
]

print("=== How numbers tokenize (GPT-4) ===")
print()
for num in number_examples:
    tokens = enc_gpt4.encode(num)
    pieces = [enc_gpt4.decode([t]) for t in tokens]
    print(f"  '{num}' → {pieces} ({len(tokens)} tokens)")

print()
print("Notice: '1000000' may be split as ['100', '0000'] or ['1000', '000']")
print("The model doesn't see '1000000' as a single number — it sees tokens.")
print("This is why LLMs are weak at exact arithmetic without a calculator tool.")
print()

# Code tokenization
code_example = """
def add(a, b):
    return a + b
"""

code_tokens = enc_gpt4.encode(code_example)
code_pieces = [enc_gpt4.decode([t]) for t in code_tokens]

print("=== How Python code tokenizes (GPT-4) ===")
print(f"Code: {repr(code_example)}")
print(f"Tokens ({len(code_tokens)} total): {code_pieces}")
print()
print("Notice: indentation (spaces) often becomes its own token.")
print("This is why consistent indentation matters even in code prompts.")

In [ ]:
# === Experiment 3.4 — Token cost calculator ===
# Let's build a simple tool to estimate the cost of a prompt

# GPT-4 pricing (approximate, as of 2024 — always check current pricing)
COST_PER_1K_TOKENS_INPUT  = 0.01   # USD per 1,000 input tokens
COST_PER_1K_TOKENS_OUTPUT = 0.03   # USD per 1,000 output tokens

def estimate_cost(prompt, expected_output_tokens=200, model='gpt-4'):
    """Estimate the API cost for a prompt + expected output."""
    # Count input tokens
    input_tokens = len(enc_gpt4.encode(prompt))
    
    # Calculate cost
    input_cost  = (input_tokens / 1000)          * COST_PER_1K_TOKENS_INPUT
    output_cost = (expected_output_tokens / 1000) * COST_PER_1K_TOKENS_OUTPUT
    total_cost  = input_cost + output_cost
    
    return {
        'input_tokens':         input_tokens,
        'expected_output_tokens': expected_output_tokens,
        'input_cost_usd':       input_cost,
        'output_cost_usd':      output_cost,
        'total_cost_usd':       total_cost,
        'cost_per_1000_calls':  total_cost * 1000,
    }

# Test with a typical system prompt
system_prompt = """
You are a helpful customer service assistant for TechCorp.
You must answer questions about our products politely and accurately.
If you don't know the answer, say "I'll check with our team and get back to you."
Never discuss competitor products. Always maintain a professional tone.
"""

user_message = "What is your return policy for electronics purchased online?"

full_prompt = system_prompt + "\nUser: " + user_message

cost_info = estimate_cost(full_prompt, expected_output_tokens=150)

print("=== Token cost estimate ===")
print()
print(f"Input tokens:            {cost_info['input_tokens']}")
print(f"Expected output tokens:  {cost_info['expected_output_tokens']}")
print(f"Input cost:              ${cost_info['input_cost_usd']:.5f}")
print(f"Output cost:             ${cost_info['output_cost_usd']:.5f}")
print(f"Total per call:          ${cost_info['total_cost_usd']:.5f}")
print(f"Cost at 1,000 calls/day: ${cost_info['cost_per_1000_calls']:.2f}/day")
print(f"Cost at 1,000 calls/day: ${cost_info['cost_per_1000_calls']*30:.2f}/month")
print()
print("This is why prompt engineering includes being concise:")
print("every unnecessary word in your system prompt costs money at scale.")

---
## SECTION 4 — CHALLENGE
### Build a code-aware BPE tokenizer and compare with tiktoken

> This section is intentionally open-ended. There are no `___` blanks — you design the solution.  
> Hints are provided if you get stuck.

In [ ]:
# === CHALLENGE: Train a BPE tokenizer on a code corpus ===

# Step 1: Define a small code corpus (word → frequency)
# These are common tokens/patterns in Python code
code_corpus = {
    'def':         120,
    'return':       95,
    'import':       80,
    'class':        70,
    'self':         200,
    'for':          150,
    'if':           180,
    'in':           160,
    'print':         90,
    'range':         85,
    'len':           75,
    'append':        60,
    'list':          65,
    'dict':          50,
    'str':           55,
    'int':           55,
    'True':          45,
    'False':         40,
    'None':          70,
    'not':           80,
    'and':           90,
    'or':            70,
    'is':            85,
    'with':          50,
    'as':            45,
}

# Step 2: Train your BPE tokenizer on this code corpus
# HINT: use your build_bpe_vocab() function from Section 2
# TODO: Train with 10 merges and store the rules

# YOUR CODE HERE


# Step 3: Tokenize these code snippets with YOUR tokenizer
code_snippets = [
    "def return import class self",
    "for if in print range len",
    "list dict str int True False None",
]

# TODO: tokenize each snippet with your BPE and print the results
# YOUR CODE HERE


# Step 4: Tokenize the same snippets with tiktoken (GPT-4)
# TODO: compare your tokenization vs tiktoken's for each snippet
# - Which tokens does your BPE merge that tiktoken doesn't?
# - Which tokens does tiktoken merge that your BPE doesn't?
# YOUR CODE HERE

print("Challenge complete!")
print("Consider: why does tiktoken's tokenization differ from yours?")
print("  - tiktoken was trained on a MUCH larger corpus (billions of documents)")
print("  - tiktoken used byte-level BPE (bytes, not characters)")
print("  - tiktoken's corpus included code, but also everything else")

In [ ]:
# === EXTENSION CHALLENGE (harder) ===
# Build a BPE tokenizer that assigns integer IDs to tokens, not just strings
#
# A real tokenizer needs:
#   token_to_id: dict mapping each token string → its integer ID
#   id_to_token: dict mapping each integer ID → its token string
#
# Design and implement:
#   encode(text) → list of integer IDs
#   decode(ids)  → string
#
# HINTS:
#   1. Start by collecting all unique symbols that appear in the final vocab.
#      These are your tokens.
#   2. Sort them and assign IDs starting from 0.
#   3. encode() = tokenize_with_bpe() then map each token to its ID
#   4. decode() = map each ID back to token string then join with spaces

class MyBPETokenizer:
    """A simple BPE tokenizer with encode/decode support."""
    
    def __init__(self):
        self.merge_rules = []    # learned during training
        self.token_to_id = {}    # token string → integer
        self.id_to_token = {}    # integer → token string
    
    def train(self, corpus_freq, num_merges):
        """Train the tokenizer on a word-frequency corpus."""
        # TODO: call build_bpe_vocab to get merge rules
        # TODO: collect all unique symbols from the final vocab
        # TODO: build token_to_id and id_to_token dictionaries
        pass
    
    def encode(self, text):
        """Convert text → list of integer token IDs."""
        # TODO: use tokenize_with_bpe then map tokens to IDs
        # Handle unknown tokens with a special <unk> ID
        pass
    
    def decode(self, ids):
        """Convert list of integer IDs → text string."""
        # TODO: map each ID to its token, join and clean up </w> markers
        pass


# Test if you implement it:
# tokenizer = MyBPETokenizer()
# tokenizer.train({'low': 5, 'lower': 2, 'newest': 6, 'widest': 3}, num_merges=8)
# ids = tokenizer.encode('low newest')
# print('Encoded:', ids)
# print('Decoded:', tokenizer.decode(ids))

print("Extension challenge ready — implement MyBPETokenizer above.")

---
## Session Wrap-Up

### What you built
- A complete BPE tokenizer from scratch (no libraries)
- Intuition for why character-level and word-level tokenization fail
- Experience comparing GPT-2 vs GPT-4 tokenization with `tiktoken`
- A token cost calculator

### Key takeaways
1. **LLMs never see raw text** — only integer IDs from a pre-agreed vocabulary
2. **BPE is a greedy algorithm** — it doesn't find the globally optimal vocabulary, just a good one
3. **More vocabulary = shorter sequences** — GPT-4's 100k vocab produces fewer tokens than GPT-2's 50k vocab for the same text
4. **Non-English text is more expensive** — the vocabulary was trained on mostly English text
5. **Token count = cost** — every word in your prompt costs money at scale

### What's next
**NB2 — Temperature, Top-K, Top-P** answers the other half of the puzzle:  
once the model has computed probabilities over the entire vocabulary, *how does it pick the next token?*

---
*Part of the GenAI-2026 curriculum — zero-to-genai-engineer track*